# Use case Moving Averages for Deep Folding projects
This use case is focused on an example of use of moving averages in deep_folding projects. <br>
Here, we suppose test skeletons have been encoded to a latent space of much fewer dimensions. <br>
We consider two cases:
- one cluster and the analysis of moving averages along one axis of this cluster,
- two clusters and the analysis of average crops of each cluster.

For this notebbok to work, you will need to point to neurospin/deep_folding folder (see root and path variable below)

WARNING: not working

### 1) Imports

In [ ]:
import pcpm
import colorado as cld
import dico_toolbox as dtx
from tqdm import tqdm

import os
import glob
import sys
import pandas as pd
import numpy as np
from scipy.spatial import distance

import plotly.graph_objects as goroot
import pickle
import matplotlib.pyplot as plt

import deep_folding.brainvisa.utils.remove_hull as rhull

from soma import aims

In [ ]:
root = '/neurospin' 
path = os.path.join(root, 'dico/data/deep_folding/history/2021_November/crops/SC/mask/sulcus_based/2mm/')
#path = os.path.join(root, 'dico/data/deep_folding/data/crops/STS_branches/sulcus_based/2mm/Rcrops/')
print(path)
print(os.path.isdir(path))

### 2) Loading of subjects's distribution in the latent space
DataFrame is obtained during analysis of beta-VAE latent space (125 dimensions) for SC crops, according to the following steps:
- loading of trained model
- encoding of test controls and asymmetry benchmark subjects
- projection of the subjects in a 2D space with tSNE algorithm
- export of points' coordinates

In [ ]:
df_subjects = pd.read_csv('../data/visualization/sub_sort_asym_bench_sc_label.csv', index_col="id_sub").drop(columns={'Unnamed: 0'})
df_subjects.index.astype(int)
subjects = df_subjects.index
df_subjects = df_subjects.rename(columns={'91': 0, '17': 1})
assert(type(list(df_subjects.index)[0])==int)
df_subjects.head()

In [ ]:
color_dict = {'normal_test': 'blue', 'benchmark': 'magenta'}

arr = np.array([np.array([df_subjects[k][i] for k in df_subjects.columns[:2]]) for i in df_subjects.index])

fig, ax = plt.subplots()
ax = fig.add_subplot(111)
for g in np.unique([df_subjects.label]):
    ix = np.where(df_subjects.label == g)
    x = [arr[ix][k][0] for k in range(len(ix[0]))]
    y = [arr[ix][k][1] for k in range(len(ix[0]))]
    if g =='benchmark':
        g_lab = 'benchmark asymmetry'
    else:
        g_lab=g
    ax.scatter(x, y, c = color_dict[g], label = g_lab)

plt.xlabel(f'1st most important feature: {df_subjects.columns[0]}', fontsize=14)
plt.ylabel(f'2nd most important feature: {df_subjects.columns[1]}', fontsize=14)
plt.show()

### 3) Construction of dictionnary of subjects' buckets

In [ ]:
print(subjects)

In [ ]:
tgt_dir = '/tmp/test/'
dataset = rhull.DatasetHullRemoved(src_dir=path,
                                   tgt_dir=tgt_dir,
                                   side='R',
                                   list_subjects=list(subjects))
buckets = dataset.create_meshes()

In [ ]:
buckets

In [ ]:
def centeroidnp(df):
    """Gives centroid of a dataframe of points.
    /!\ Centroid is not a point within the defined set of points
    
    Args:
        IN: df: pandas.DataFrame with index corresponding to subjects and as many columns as coordinates
        OUT: coords: tuple of coordinates of centroid
    """
    coords = []
    ndim = len(df.columns) # number of dimensions to consider
    
    for k in range(len(df.columns)):
        arr = np.array(list(df[df.columns[k]]))
        length = arr.shape[0]
        sum_k = np.sum(arr)
        coords.append(sum_k/length)

    return tuple(coords)

In [ ]:
def closest_distance(centroid, df):
    """Returns closest point to centroid of a given cluster
    /!\ central_point is part of the defined set of points
    
    Args:
        IN: centroid: tuple of coordinates of centroid
            df: pandas.DataFrame with index corresponding to subjects and as many columns as coordinates
        OUT: central_point: subject ID corresponding to closest point to the centroid
    """
    # df must have as many columns as dimensions to consider
    distances = {}
    # Get distances to centroid for each point
    for sub in list(df.index):
        pos = [df.loc[sub][k] for k in range(len(df.columns))]
        distances[sub] = distance.euclidean(pos, centroid)
        
    # Get closest point to centroid
    central_point = min(distances, key=distances.get)
    return central_point

### 4) One cluster analysis

#### Get central subject

In [ ]:
centroid = centeroidnp(df_subjects[[0, 1]])
print(f"Coordinates of centroid are: {centroid}")
central = closest_distance(centroid, df_subjects[[0, 1]])
print(f"Closest subject to centroid is {central}")

#### Alignment of all subjects to central subject

In [ ]:
aligned_buckets, aligned_rot, aligned_transl = pcpm.calc_MA_volumes_with_alignment(buckets, central, cores=2)

In [ ]:
#fig = cld.draw_numpy_buckets(list(aligned_buckets.values()), shift=(0,50,0))
#ma.plot.brochette_layout(fig)

#### Subjects histogram

In [ ]:
import seaborn as sns
sns.displot(df_subjects, x=0)

In [ ]:
buckets

#### Creation of moving averages

In [ ]:
SPAM_centers = np.arange(-3, 2, 1).astype(float) # hist des sujets => centres des moving average
# SPAM_centers = [1.]
SPAM_centers

In [ ]:
# moving_averages doesn't like string label
df_subjects['label'][:] = 0

In [ ]:
df_subjects.head()

In [ ]:
print(type(SPAM_centers))
print(type(aligned_buckets))

In [ ]:
SPAM_vols, shift = pcpm.calc_MA_volumes_batch(SPAM_centers, aligned_buckets, df_subjects, axis_n=0, FWHM=1)

In [ ]:
from tqdm import tqdm
SPAM_meshes = {}

for k, volume in tqdm(list(SPAM_vols.items())[:]):
    SPAM_meshes[k]=dtx.convert.volume_to_mesh(
                    vol=volume, smoothRate=0.3,
                    threshold="90%")
    
shifted_SPAM_meshes = {}
for dist, mesh in SPAM_meshes.items():
    shifted_SPAM_meshes[str(dist)]=dtx.mesh.shift_aims_mesh_along_axis(mesh, 20*dist)

#### Visualization of MA

In [ ]:
fig = cld.draw(shifted_SPAM_meshes)
ma.plot.brochette_layout(fig, "subjects meshes")

#### Saving of MA

In [ ]:
# for x, mesh in tqdm(shifted_SPAM_meshes.items()):
#     aims.write(mesh, f"MA_{x}_sc.mesh")

### 5) 2 clusters analysis

#### Loading of subjects of the two clusters

In [ ]:
tsne_subjects = pd.read_csv('../data/visualization/tsne_asym_bench_sc.csv', index_col="id_sub").drop(columns={'Unnamed: 0'})
subjects_tsne = tsne_subjects.index
tsne_subjects = tsne_subjects.rename(columns={'tsne: 0': 0, 'tsne: 1': 1})
tsne_subjects.head()

We can use labels if we have them (in the case where we use benchmark subjects vs controls for example):

In [ ]:
color_dict = {'normal_test': 'blue', 'benchmark': 'magenta'}

arr = np.array([np.array([tsne_subjects[k][i] for k in tsne_subjects.columns[:2]]) for i in tsne_subjects.index])

fig, ax = plt.subplots()
ax = fig.add_subplot(111)
for g in np.unique([tsne_subjects.label]):
    ix = np.where(tsne_subjects.label == g)
    x = [arr[ix][k][0] for k in range(len(ix[0]))]
    y = [arr[ix][k][1] for k in range(len(ix[0]))]
    if g =='benchmark':
        g_lab = 'benchmark asymmetry'
    else:
        g_lab=g
    ax.scatter(x, y, c = color_dict[g], label = g_lab)

plt.xlabel(f'1st most important feature: {tsne_subjects.columns[0]}', fontsize=14)
plt.ylabel(f'2nd most important feature: {tsne_subjects.columns[1]}', fontsize=14)
plt.show()

We can also imagine a case where we don't have any labels but two clusters and apply a kmeans algorithm:

In [ ]:
from sklearn.cluster import KMeans

X = np.array([np.array([tsne_subjects[k][i] for k in tsne_subjects.columns[:2]]) for i in tsne_subjects.index])

kmeans = KMeans(n_clusters=2, random_state=0).fit(X)

labels = kmeans.labels_
tsne_subjects['kmeans_label'] = labels
clusters_centroids = kmeans.cluster_centers_
print(f"cluster's centroids coordinates: \n {clusters_centroids}")

In [ ]:
color_dict = {1: 'blue', 0: 'magenta'}

arr = np.array([np.array([tsne_subjects[k][i] for k in tsne_subjects.columns[:2]]) for i in tsne_subjects.index])

fig, ax = plt.subplots()
ax = fig.add_subplot(111)

for g in np.unique(kmeans.labels_):
    ix = np.where(kmeans.labels_ == g)
    x = [arr[ix][k][0] for k in range(len(ix[0]))]
    y = [arr[ix][k][1] for k in range(len(ix[0]))]
    if g =='benchmark':
        g_lab = 'benchmark asymmetry'
    else:
        g_lab=g
    ax.scatter(x, y, c = color_dict[g], label = g_lab)

ax.scatter(clusters_centroids[0][0], clusters_centroids[0][1], color='crimson', marker='X')
ax.scatter(clusters_centroids[1][0], clusters_centroids[1][1], color='navy', marker='X')
ax.scatter(tsne_subjects[0][116726], tsne_subjects[1][116726], color='forestgreen')
ax.scatter(tsne_subjects[0][163836], tsne_subjects[1][163836], color='forestgreen')


plt.xlabel(f'{tsne_subjects.columns[0]}', fontsize=14)
plt.ylabel(f'{tsne_subjects.columns[1]}', fontsize=14)
plt.show()

#### Central subjects for both clusters:

In [ ]:
central_1 = closest_distance(clusters_centroids[0], tsne_subjects.drop(['label', 'kmeans_label'], axis=1))
print(f"Closest subject to centroid of cluster 1 is {central_1}")
central_2 = closest_distance(clusters_centroids[1], tsne_subjects.drop(['label', 'kmeans_label'], axis=1))
print(f"Closest subject to centroid of cluster 2 is {central_2}")

We create sub-dataframes based on labels

In [ ]:
cluster1 = tsne_subjects[tsne_subjects.kmeans_label==0]
cluster2 = tsne_subjects[tsne_subjects.kmeans_label==1]
assert(len(np.unique(list(cluster1.kmeans_label)))==1)
assert(len(np.unique(list(cluster2.kmeans_label)))==1)

#### Creation of buckets dictionnary

In [ ]:
subjects_c1 = cluster1.index
subjects_c2 = cluster2.index

buckets_c1 = {k: v for k,v in buckets.items() if k in list(subjects_c1)}
buckets_c2 = {k: v for k,v in buckets.items() if k in list(subjects_c2)}

#### Alignement of the subjects to respective central subject

In [ ]:
aligned_buckets_C1, aligned_rot_C1, aligned_transl_C1 = ma.align_buckets_by_ICP_batch(buckets_c1, central_1, cores=2)
aligned_buckets_C2, aligned_rot_C2, aligned_transl_C2 = ma.align_buckets_by_ICP_batch(buckets_c2, central_2, cores=2)

In [ ]:
sns.displot(cluster1, x=0)

In [ ]:
sns.displot(cluster2, x=0)

In [ ]:
cluster1['label'][:] = 0
cluster1.head()

In [ ]:
SPAM_centers_c1 = [0]
SPAM_vols_c1, shift1 = ma.calc_MA_volumes_batch(SPAM_centers_c1, aligned_buckets_C1, cluster1, axis_n=0, FWHM=1)

In [ ]:
cluster2['label'][:] = 0

In [ ]:
SPAM_centers_c2 = [4.5]
SPAM_vols_c2, shift2 = ma.calc_MA_volumes_batch(SPAM_centers_c2, aligned_buckets_C2, cluster2, axis_n=0, FWHM=1)

In [ ]:
SPAM_meshes = {}

for k, volume in tqdm(list(SPAM_vols_c1.items())[:]+list(SPAM_vols_c2.items())[:]):
    SPAM_meshes[k]=dtx.convert.volume_to_mesh(
                    vol=volume, smoothRate=0.2,
                    threshold="90%")
    
shifted_SPAM_meshes = {}
for dist, mesh in SPAM_meshes.items():
    shifted_SPAM_meshes[str(dist)]=dtx.mesh.shift_aims_mesh_along_axis(mesh, 10*dist)

#### Visualization of average crops of both clusters

In [ ]:
fig = cld.draw(shifted_SPAM_meshes)
ma.plot.brochette_layout(fig, "subjects meshes")

#### Saving of average crops

In [ ]:
# for x, mesh in tqdm(shifted_SPAM_meshes.items()):
#     aims.write(mesh, f"MA_{x}_2cluster.mesh")